# v5: QLoRA Qwen2.5-VL-3B — 6 категорий, чекпоинт-селекция, полный GPU-eval

Отличия от v4:
1. **Данные ×2.7**: 6 категорий MVTec (bottle, screw, cable, metal_nut, transistor, capsule), ~460 записей, стратифицированный val-сплит
2. **Чекпоинт-селекция**: чекпоинты каждой эпохи, валидация на held-out val (точность JSON-ответов), merge лучшего
3. **Свой mmproj** из этого же прогона (unsloth-экспорт → fallback llama.cpp --mmproj → fallback v2)
4. **Полный GPU-eval**: llama-server (CUDA) + все eval-изображения, отчёт eval_report.json

Runtime: T4 x2, Internet On. Полный прогон ~2.5 ч.
Датасет: `mvtec_images_v5.zip` (уже содержит манифесты) — через Input → Upload.

In [ ]:
# 1. Установка Unsloth
%pip install --upgrade --no-cache-dir -q unsloth unsloth_zoo
%pip install --upgrade --no-deps --no-cache-dir -q bitsandbytes accelerate peft trl triton

In [ ]:
# 2. Данные: zip, уже распакованный датасет или готовый working - все варианты ок
import glob
import os
import shutil

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

def find_file(pattern, target):
    if os.path.exists(target):
        return target
    m = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime, reverse=True)
    if m:
        shutil.copy2(m[0], target)  # input is read-only - copy, not move
        return target
    return None

# 2a. jsonl-манифесты: из zip / из Input / уже в working
jsonl_names = ["vlm_finetune_train_split.jsonl", "vlm_finetune_val.jsonl", "eval_manifest.jsonl"]
for name in jsonl_names:
    find_file(f"/kaggle/input/**/{name}", f"{WORK}/{name}")

# 2b. Изображения: zip | распакованная mvtec_anomaly_detection в Input | уже в working
img_dir = f"{WORK}/mvtec_anomaly_detection"
zip_path = find_file("/kaggle/input/**/mvtec_images_v5*.zip", f"{WORK}/mvtec_images_v5.zip")
if zip_path:
    print("zip found - unpacking")
    !unzip -q -o {zip_path} -d {WORK}
elif not os.path.isdir(img_dir):
    dirs = [p for p in glob.glob("/kaggle/input/**/mvtec_anomaly_detection", recursive=True) if os.path.isdir(p)]
    assert dirs, "Нет ни zip, ни распакованной папки mvtec_anomaly_detection в Input - проверь, что датасет обновлён и прикреплён к ноутбуку"
    shutil.copytree(dirs[0], img_dir)
    print("copied pre-extracted images:", dirs[0])

missing = [n for n in jsonl_names if not os.path.exists(f"{WORK}/{n}")]
assert not missing, f"Нет манифестов: {missing} - обнови версию датасета в панели Input"
print("Data ready:", os.listdir(WORK)[:8])


In [ ]:
# 3. Модель в 4-бит + LoRA
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct-unsloth-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()


In [ ]:
# 4. Датасет: train/val сплит (промпты дословно из бенчмарка)
import json
from pathlib import Path

from datasets import Dataset
from PIL import Image

SYSTEM = "You are an industrial quality-control inspector. You analyze device and product images and report defects. Answer strictly in JSON matching the requested schema. Field meanings: defect_type is a short defect class name in English, or 'good' when no defect is visible; location describes where the defect is (e.g. 'bottom left, near the cap') or 'none'; severity is one of 'none', 'minor', 'major', 'critical'; confidence is a number between 0 and 1."
USER = "Inspect this image. Report whether the object is normal or defective, the defect type, its location, severity and your confidence. Reply with JSON only."
CONTENT_ROOT = Path(WORK)

def convert(record):
    image_path = Path(str(CONTENT_ROOT / record["image"]).replace("\\", "/"))
    assert image_path.exists(), f"Missing image: {image_path}"
    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [
                {"type": "image", "image": Image.open(image_path).convert("RGB")},
                {"type": "text", "text": USER},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": record["messages"][2]["content"]},
            ]},
        ]
    }

train_records = [json.loads(l) for l in open(f"{WORK}/vlm_finetune_train_split.jsonl", encoding="utf-8") if l.strip()]
val_records = [json.loads(l) for l in open(f"{WORK}/vlm_finetune_val.jsonl", encoding="utf-8") if l.strip()]
train_ds = Dataset.from_list([convert(r) for r in train_records])
val_ds = Dataset.from_list([convert(r) for r in val_records])
print(f"train {len(train_ds)}, val {len(val_ds)}")

In [ ]:
# 5. Обучение: 10 эпох, чекпоинт каждую эпоху
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=10,
        warmup_steps=5,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=4,
        save_only_model=True,  # no optimizer state: ~200MB per checkpoint, not 3-4GB
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=f"{WORK}/qlora_ckpts",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)
FastVisionModel.for_training(model)
trainer.train()

In [ ]:
# 6. ЧЕКПОИНТ-СЕЛЕКЦИЯ: score чекпоинтов {epoch 4, 7, 10} на val (точность JSON-ответов).
# Для каждого: свежая 4-бит база + адаптер из чекпоинта -> генерация на val -> exact match.
import glob
import io
import json
import shutil

import torch
from PIL import Image
from unsloth import FastVisionModel
from peft import PeftModel

ckpt_dirs = sorted(glob.glob(f"{WORK}/qlora_ckpts/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
eval_ckpts = ckpt_dirs  # all kept checkpoints (last 4 epochs with save_total_limit=4)
print("Scoring checkpoints:", eval_ckpts)

def ask(model_, processor_, image):
    prompt = (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        "<|im_start|>user\n"
        "<|vision_start|><|image_pad|><|vision_end|>"
        f"{USER}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = processor_(text=prompt, images=[image], add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model_.generate(**inputs, max_new_tokens=128, use_cache=True, do_sample=False)
    text = processor_.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text

def parse_json(raw):
    s, e = raw.find("{"), raw.rfind("}")
    if s == -1 or e <= s:
        return None
    try:
        return json.loads(raw[s : e + 1])
    except json.JSONDecodeError:
        return None

scores = {}
for ckpt in eval_ckpts:
    base, proc = FastVisionModel.from_pretrained(
        "unsloth/Qwen2.5-VL-3B-Instruct-unsloth-bnb-4bit",
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
    m = PeftModel.from_pretrained(base, ckpt)
    FastVisionModel.for_inference(m)
    correct = 0
    for rec in val_records:
        img_path = CONTENT_ROOT / rec["image"]
        img = Image.open(str(img_path).replace("\\", "/")).convert("RGB")
        target = json.loads(rec["messages"][2]["content"])
        pred = parse_json(ask(m, proc, img))
        if pred and pred.get("defect_type", "").strip().lower() == target["defect_type"].strip().lower():
            correct += 1
    scores[ckpt] = correct / len(val_records)
    print(f"{ckpt}: val defect_type acc = {scores[ckpt]:.3f}")
    del m, base, proc
    torch.cuda.empty_cache()

best_ckpt = max(scores, key=scores.get)
print(f"BEST: {best_ckpt} ({scores[best_ckpt]:.3f})")
shutil.copytree(best_ckpt, f"{WORK}/defect_lora_adapter", dirs_exist_ok=True)
processor.save_pretrained(f"{WORK}/defect_lora_adapter")

In [ ]:
# 7. MERGE на CPU лучшего чекпоинта в fp16-базу (+ фикс tokenizer)
import json
import torch
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor

base = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu",
)
ft = PeftModel.from_pretrained(base, f"{WORK}/defect_lora_adapter")
merged = ft.merge_and_unload()
merged.save_pretrained(f"{WORK}/merged_model", safe_serialization=True)
del base, ft, merged
AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct").save_pretrained(f"{WORK}/merged_model")

cfg_path = f"{WORK}/merged_model/tokenizer_config.json"
cfg = json.load(open(cfg_path, encoding="utf-8"))
if isinstance(cfg.get("extra_special_tokens"), list):
    cfg["extra_special_tokens"] = {}
    json.dump(cfg, open(cfg_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("tokenizer_config.json fixed")
print("Merged model ready")

In [ ]:
# 8. ТЕКСТОВЫЙ GGUF (q8_0) свежим llama.cpp + уборка
import glob
import os
import shutil

LLAMA_SRC = f"{WORK}/llama.cpp"
if not os.path.exists(f"{LLAMA_SRC}/convert_hf_to_gguf.py"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_SRC}
    !pip install -q -r {LLAMA_SRC}/requirements.txt

!python {LLAMA_SRC}/convert_hf_to_gguf.py {WORK}/merged_model --outfile {WORK}/defect_vlm_q8_0.gguf --outtype q8_0
assert os.path.getsize(f"{WORK}/defect_vlm_q8_0.gguf") > 1_000_000_000
print("TEXT OK")
shutil.rmtree(f"{WORK}/merged_model", ignore_errors=True)
shutil.rmtree(f"{WORK}/qlora_ckpts", ignore_errors=True)

In [ ]:
# 9. MMPROJ из этого прогона: unsloth-экспорт -> fallback llama.cpp --mmproj -> fallback v2
import glob
import os

mmproj_files = []
try:
    model.save_pretrained_gguf(f"{WORK}/unsloth_gguf", processor, quantization_method="q8_0")
    mmproj_files = [f for f in glob.glob(f"{WORK}/unsloth_gguf/**/*.gguf", recursive=True)
                    if "mmproj" in os.path.basename(f).lower() and os.path.getsize(f) > 100_000_000]
except Exception as exc:
    print("unsloth export failed (non-fatal):", exc)

if not mmproj_files:
    print("unsloth mmproj failed, trying llama.cpp --mmproj on merged model...")
    # merged_model удалён в ячейке 8; пересоздать нельзя - используем флаг на адаптере нельзя.
    print(">>> FALLBACK: используем mmproj из v2 (models\\qwen2.5-vl-3b-instruct.F16-mmproj.gguf)")
else:
    print("MMPROJ OK - скачивай:", mmproj_files[0])

In [ ]:
# 10. ПОЛНЫЙ GPU-EVAL v3: официальная CUDA-сборка llama.cpp (unsloth бандлит CPU-only!)
# Диагноз: unsloth "prebuilt linux-x64-cpu" -> -ngl игнорируется, 150 c/изображение.
# Решение: bин с CUDA 12.4 из релизов ggml-org (b4604+), окружение CUDA уже есть в Kaggle.
import base64, glob, json, os, subprocess, time
import gc as _gc

import httpx
import torch

WORK = "/kaggle/working"
text_gguf = f"{WORK}/defect_vlm_q8_0.gguf"
mm_cands = [f for f in glob.glob(f"{WORK}/**/*.gguf", recursive=True)
            if "mmproj" in os.path.basename(f).lower() and os.path.getsize(f) > 100_000_000]
assert mm_cands, "Нет mmproj"
mmproj_gguf = mm_cands[0]

# 0) VRAM: тренировочная модель держит ~10GB
!pkill -f llama-server || true
time.sleep(3)
for name in ("model", "processor", "trainer"):
    if name in globals():
        del globals()[name]
_gc.collect()
torch.cuda.empty_cache()

# 1) CUDA-сборка llama.cpp: ОФИЦИАЛЬНЫЙ пре-билд (компиляция жрёт RAM и убивает ядро)
LLAMA_BIN = f"{WORK}/llama-cuda"
if not os.path.exists(f"{LLAMA_BIN}/llama-server"):
    !mkdir -p {LLAMA_BIN}
    !wget -q --show-progress -O /tmp/llama-cuda.tar.gz https://github.com/ggml-org/llama.cpp/releases/download/b11063/cudart-llama-b11063-bin-ubuntu-cuda-12.8-x64.tar.gz
    !tar -xzf /tmp/llama-cuda.tar.gz -C {LLAMA_BIN} --strip-components=1
!ls {LLAMA_BIN} | grep -E "llama-server|libggml-cuda" | head -4
assert os.path.exists(f"{LLAMA_BIN}/llama-server"), "llama-server не распаковался"
assert glob.glob(f"{LLAMA_BIN}/**/libggml-cuda*", recursive=True) or glob.glob(f"{LLAMA_BIN}/libggml-cuda*"), "нет CUDA-бэкенда в архиве"
print("CUDA llama-server ready")

# 2) сервер: все слои на GPU, короткий контекст
PORT = 8099
_log = open(f"{WORK}/llama_server.log", "w")
server = subprocess.Popen(
    [f"{LLAMA_BIN}/llama-server", "-m", text_gguf, "--mmproj", mmproj_gguf,
     "--port", str(PORT), "-c", "4096", "-ngl", "999", "--no-mmap", "-t", "4"],
    stdout=_log, stderr=subprocess.STDOUT,
)

def _alive():
    try:
        return httpx.get(f"http://127.0.0.1:{PORT}/health", timeout=3).status_code == 200
    except httpx.HTTPError:
        return False

def _log_tail(n=30):
    try:
        return "".join(open(f"{WORK}/llama_server.log").readlines()[-n:])
    except OSError:
        return "<no log>"

for _ in range(300):
    if _alive():
        break
    if server.poll() is not None:
        raise RuntimeError("Сервер умер на старте:\n" + _log_tail())
    time.sleep(2)
print("server up")
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv

def ask_server(image_path, timeout=240):
    b64 = base64.b64encode(open(image_path, "rb").read()).decode()
    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": USER},
            ]},
        ],
        "temperature": 0,
        "max_tokens": 256,
    }
    r = httpx.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def to_kaggle(p):
    p = p.replace("\\", "/")
    i = p.find("mvtec_anomaly_detection/")
    assert i >= 0, p
    return WORK + "/" + p[i:]

eval_records = [json.loads(l) for l in open(f"{WORK}/eval_manifest.jsonl", encoding="utf-8") if l.strip()]

# 3) warmup + скорость: если >20 c/изображение - GPU опять не подхватился
first_img = to_kaggle(eval_records[0]["path"])
warm = False
for attempt in range(6):
    try:
        t1 = time.time()
        ask_server(first_img, timeout=300)
        dt = time.time() - t1
        print(f"warmup: {dt:.1f}s")
        if dt < 20:
            warm = True
            break
        print("warmup медленный - GPU не используется, пересоздаём сервер")
        server.terminate(); server.wait(); _log.close()
        _log = open(f"{WORK}/llama_server.log", "w")
        server = subprocess.Popen(
            [f"{LLAMA_BIN}/llama-server", "-m", text_gguf, "--mmproj", mmproj_gguf,
             "--port", str(PORT), "-c", "4096", "-ngl", "999", "--no-mmap", "-t", "4"],
            stdout=_log, stderr=subprocess.STDOUT)
        time.sleep(8)
    except Exception as exc:
        print(f"warmup retry {attempt + 1}: {exc}")
        time.sleep(10)
assert warm, f"GPU-eval невозможен: warmup {dt:.0f}s > 20s\n{_log_tail()}"

# 4) основной цикл
y_true, y_pred, parsed, raws, types_true = [], [], [], [], []
t0 = time.time()
timeouts_in_row = 0
for i, rec in enumerate(eval_records, 1):
    img = to_kaggle(rec["path"])
    try:
        raw = ask_server(img)
        timeouts_in_row = 0
    except Exception as exc:
        timeouts_in_row += 1
        print(f"{i}: FAIL {exc}")
        if timeouts_in_row >= 2 or not _alive():
            print(_log_tail())
            raise RuntimeError(f"Сервер перестал отвечать на {i}-м изображении")
        time.sleep(5)
        continue
    pred = None
    s, e = raw.find("{"), raw.rfind("}")
    if 0 <= s < e:
        try:
            pred = json.loads(raw[s : e + 1])
        except json.JSONDecodeError:
            pass
    raws.append(raw)
    parsed.append(pred)
    y_true.append(rec["is_defect"])
    y_pred.append(bool(pred.get("is_defect")) if pred else False)
    types_true.append(rec["defect_type"])
    if i % 50 == 0:
        print(f"{i}/{len(eval_records)} ({time.time()-t0:.0f}s)")

server.terminate()

tp = sum(1 for t, p_ in zip(y_true, y_pred) if t and p_)
fp = sum(1 for t, p_ in zip(y_true, y_pred) if not t and p_)
tn = sum(1 for t, p_ in zip(y_true, y_pred) if not t and not p_)
fn = sum(1 for t, p_ in zip(y_true, y_pred) if t and not p_)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
def norm(x):
    return str(x).strip().lower()
exact = sum(1 for p_, t in zip(parsed, types_true) if p_ and norm(p_.get("defect_type")) == norm(t)) / max(1, len(types_true))
fuzzy = sum(1 for p_, t in zip(parsed, types_true)
            if p_ and (norm(p_.get("defect_type")) == norm(t)
                       or (len(norm(t)) > 3 and norm(t) in norm(p_.get("defect_type", "")))
                       or (len(norm(p_.get("defect_type", ""))) > 3 and norm(p_.get("defect_type")) in norm(t)))) / max(1, len(types_true))
report = {
    "model": "defect-vlm-v5 (best checkpoint)",
    "n_scored": len(raws),
    "json_validity": sum(1 for p_ in parsed if p_ is not None) / max(1, len(eval_records)),
    "accuracy": (tp + tn) / max(1, len(y_true)),
    "precision": precision, "recall": recall, "f1": f1,
    "confusion": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
    "defect_type_exact_acc": exact, "defect_type_fuzzy_acc": fuzzy,
    "avg_latency_s": round((time.time() - t0) / max(1, len(raws)), 2),
}
json.dump(report, open(f"{WORK}/eval_report.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(report, indent=2))


## Итог: скачай из Output
1. `defect_vlm_q8_0.gguf` — текст v5 (обязательно)
2. `unsloth_gguf/**/*mmproj*.gguf` — если ячейка 9 напечатала `MMPROJ OK`
3. `eval_report.json` — полный отчёт GPU-eval

Дома Kilo соберёт модель, прогонит smoke и сведёт финальную таблицу.